In [26]:
from dotenv import load_dotenv
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

load_dotenv()

# Verificar claves API críticas
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY no encontrada en .env")
if not os.getenv("LANGSMITH_API_KEY"):
    raise ValueError("LANGSMITH_API_KEY no encontrada en .env")

In [27]:
from langchain_core.prompts import ChatPromptTemplate

LEGAL_JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
Eres un experto legal especializado en legislación boliviana.

Recibirás:

1. Pregunta legal del usuario.
2. Respuesta de referencia (base de verdad).
3. Respuesta del modelo.

Evalúa la respuesta del modelo de acuerdo con los estándares legales bolivianos.

Verifica:

1. **Exactitud normativa:** ¿coincide con la ley boliviana? (Revisalo únicamente con el contexto de la respuesta de referencia)
2. **Calidad del razonamiento jurídico.**
3. **Presencia de leyes o artículos alucinados (inventados).**
4. **Riesgo de asesoramiento legal engañoso.**

**Calificación:**

* Exactitud Normativa: 1–5
* Razonamiento Jurídico: 1–5
* Citación Correcta: verdadero/falso
* Alucinación Detectada: verdadero/falso

**Nivel de Riesgo:**

* BAJO (LOW)
* MEDIO (MEDIUM)
* ALTO (HIGH)

Devuelve el resultado en formato **JSON**:

```json
{{
  "normative_accuracy": int,
  "legal_reasoning": int,
  "hallucination": true/false,
  "risk_level": "LOW|MEDIUM|HIGH",
  "overall_pass": true/false,
  "justification": "breve explicación"
}}
```
"""),
    ("human", """
Question:
{question}

Reference Answer:
{reference_answer}

Model Answer:
{model_answer}
""")
])


In [28]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage


async def no_rag_legal_model(inputs: dict) -> dict:
    user_input = inputs["question"]

    # Modelo base sin RAG, sin contexto normativo
    llm = ChatOllama(model="qwen3:8b", temperature=0.7)
    
    result = await llm.ainvoke([HumanMessage(content=user_input)])

    return {
        "output": result.content
    }

In [29]:
#await rag_legal_model(inputs={'question':'prueba'})

In [30]:
from pydantic import BaseModel, Field
from typing import Literal

class LegalEvaluation(BaseModel):
    normative_accuracy: int = Field(ge=1, le=5)
    legal_reasoning: int = Field(ge=1, le=5)
    hallucination: bool
    risk_level: Literal["LOW", "MEDIUM", "HIGH"]
    overall_pass: bool
    justification: str


In [31]:
from langchain_openai import ChatOpenAI
judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

structured_llm = judge_llm.with_structured_output(LegalEvaluation)

judge_chain = LEGAL_JUDGE_PROMPT | structured_llm


In [24]:
#structured_judge.invoke([SystemMessage(judge_prompt),HumanMessage("prueba")])

In [32]:
async def legal_judge_evaluator(inputs, outputs, reference_outputs):
    question = inputs["question"]
    reference = reference_outputs["answer"]
    model_answer = outputs["output"]

    evaluation = await judge_chain.ainvoke({
        "question": question,
        "reference_answer": reference,
        "model_answer": model_answer
    })

    base_score = (
        (evaluation.normative_accuracy * 0.4) +
        (evaluation.legal_reasoning * 0.4) +
        1
    )

    overall_score = (base_score / 5) * 10

    if evaluation.hallucination:
        overall_score -= 1

    if evaluation.risk_level == "HIGH":
        overall_score -= 2
    elif evaluation.risk_level == "MEDIUM":
        overall_score -= 1

    overall_score = round(max(1, min(10, overall_score)), 2)

    return [
        {"key": "overall_score_1_10",  "score": overall_score},
        {"key": "normative_accuracy",  "score": evaluation.normative_accuracy},
        {"key": "legal_reasoning",     "score": evaluation.legal_reasoning},
        {"key": "hallucination",       "score": int(evaluation.hallucination)},
        {"key": "risk_level",          "score": {"LOW": 0, "MEDIUM": 1, "HIGH": 2}[evaluation.risk_level]},
    ]


In [33]:
from langsmith.evaluation import aevaluate
from langsmith import Client

results = await aevaluate(
    no_rag_legal_model,
    data='Kantuta AI Evaluation Dataset v6',
    evaluators=[legal_judge_evaluator],
    experiment_prefix="kantuta-ai-eval-v13"
)



LangSmithError: Failed to GET /datasets in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/datasets?limit=1&name=Kantuta+AI+Evaluation+Dataset+v6', '{"detail":"Forbidden"}')